# block-group-stack — faded example 3: Complete the shape-changer index computation in a BlockGroup audit

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `block-group-stack`. Running the beacon reports progress on the `CNN: BlockGroup stack` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BlockGroup stack` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`block-group-stack`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "block-group-stack"
DD_SUBTOPIC = "CNN: BlockGroup stack"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A block is a *shape changer* if it alters the tensor shape - i.e. `in_feats != out_feats` (channel change) OR `first_stride != 1` (spatial downsample). A canonical BlockGroup has exactly one shape changer, at index 0; the audit lists their indices to check that invariant.

## Faded exercise 3

### Complete the audit's shape-changer detector

The audit walks a `nn.Sequential` of `ResBlock`s. Everything is written except the comprehension that finds the indices of the shape-changing blocks. Fill it in.

**Fill in:** Computing the list of indices i where block i has in_feats != out_feats OR first_stride != 1.

In [ ]:
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, in_feats, out_feats, first_stride=1):
        super().__init__()
        self.in_feats = in_feats
        self.out_feats = out_feats
        self.first_stride = first_stride
        self.proj = nn.Conv2d(in_feats, out_feats, kernel_size=1, stride=first_stride)
    def forward(self, x):
        return self.proj(x)

def audit_group(group):
    blocks = list(group)
    n = len(blocks)
    shape_changers = [
        i for i, b in enumerate(blocks)
        if b.in_feats != b.out_feats or b.first_stride != 1
    ]
    return {
        'n_blocks': n,
        'shape_changers': shape_changers,
        'is_canonical': n > 0 and shape_changers == [0],
    }

import torch.nn as nn

def _test():
    # canonical group
    good = nn.Sequential(
        ResBlock(8, 16, first_stride=2),
        ResBlock(16, 16, first_stride=1),
        ResBlock(16, 16, first_stride=1),
    )
    a = audit_group(good)
    assert a['n_blocks'] == 3
    assert a['shape_changers'] == [0]
    assert a['is_canonical'] is True
    # miswired: downsample in the middle
    bad = nn.Sequential(
        ResBlock(8, 16, first_stride=1),
        ResBlock(16, 16, first_stride=2),
        ResBlock(16, 16, first_stride=1),
    )
    b = audit_group(bad)
    assert b['shape_changers'] == [0, 1], b['shape_changers']
    assert b['is_canonical'] is False
    # single shape-changing block
    one = nn.Sequential(ResBlock(4, 4, first_stride=1))
    c = audit_group(one)
    assert c['shape_changers'] == []
    assert c['is_canonical'] is False

try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, in_feats, out_feats, first_stride=1):
        super().__init__()
        self.in_feats = in_feats
        self.out_feats = out_feats
        self.first_stride = first_stride
        self.proj = nn.Conv2d(in_feats, out_feats, kernel_size=1, stride=first_stride)
    def forward(self, x):
        return self.proj(x)

def audit_group(group):
    blocks = list(group)
    n = len(blocks)
    shape_changers = [
        i for i, b in enumerate(blocks)
        if b.in_feats != b.out_feats or b.first_stride != 1
    ]
    return {
        'n_blocks': n,
        'shape_changers': shape_changers,
        'is_canonical': n > 0 and shape_changers == [0],
    }
```
</details>